In [ ]:

import numpy as np

from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score


# ==========================================
# STEP 1: HARD VOTING
# ==========================================

def hard_vote(preds):
    preds = np.asarray(preds)

    n_samples = preds.shape[1]

    result = []

    for i in range(n_samples):
        counts = np.bincount(preds[:, i])
        result.append(np.argmax(counts))

    return np.array(result)


# ==========================================
# STEP 2: SOFT VOTING
# ==========================================

def soft_vote(probas):
    probas = np.asarray(probas)

    avg_probas = np.mean(probas, axis=0)

    return np.argmax(avg_probas, axis=1)


# ==========================================
# STEP 3: GENERATE DATA
# ==========================================

def make_data(seed=42):

    X, y = make_classification(
        n_samples=900,
        n_features=20,
        n_informative=8,
        n_redundant=4,
        random_state=seed
    )

    return X, y


# ==========================================
# STEP 4: SINGLE TREE VS BAGGING
# ==========================================

def compare_single_vs_bagging(X, y, seed=0):

    base = DecisionTreeClassifier(
        random_state=seed
    )

    bagging = BaggingClassifier(
        estimator=base,
        n_estimators=50,
        random_state=seed
    )

    # Cross-validation for single decision tree
    single_scores = cross_val_score(
        base,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )

    # Cross-validation for bagging classifier
    bagging_scores = cross_val_score(
        bagging,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )

    single_acc = single_scores.mean()
    bagging_acc = bagging_scores.mean()

    return float(single_acc), float(bagging_acc)


# ==========================================
# STEP 5: RANDOM FOREST OOB SCORE
# ==========================================

def rf_oob(X, y, n_estimators=300, seed=0):

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        oob_score=True,
        bootstrap=True,
        random_state=seed
    )

    model.fit(X, y)

    return float(model.oob_score_)


# ==========================================
# STEP 6: VERIFY HARD VOTING
# ==========================================

preds = np.array([
    [0, 1, 2, 2],
    [0, 1, 1, 2],
    [1, 1, 1, 0]
])

print("Hard Voting:")
print(hard_vote(preds))

# Expected: [0 1 1 2]


# ==========================================
# STEP 7: VERIFY SOFT VOTING
# ==========================================

probas = np.array([
    [
        [0.8, 0.2],
        [0.4, 0.6]
    ],
    [
        [0.7, 0.3],
        [0.3, 0.7]
    ]
])

print("\nSoft Voting:")
print(soft_vote(probas))

# Expected: [0 1]


# ==========================================
# STEP 8: VERIFY DATA GENERATION
# ==========================================

X, y = make_data(42)

print("\nDataset Shape:")
print("X:", X.shape)
print("y:", y.shape)

# Expected:
# X: (900, 20)
# y: (900,)


# ==========================================
# STEP 9: COMPARE SINGLE TREE AND BAGGING
# ==========================================

s, b = compare_single_vs_bagging(X, y)

print("\nSingle Decision Tree Accuracy:")
print(round(s, 3))

print("\nBagging Accuracy:")
print(round(b, 3))


# ==========================================
# STEP 10: RANDOM FOREST OOB ACCURACY
# ==========================================

oob = rf_oob(X, y, n_estimators=200)

print("\nRandom Forest OOB Accuracy:")
print(round(oob, 3))


# ==========================================
# STEP 11: TEST TIE BREAKING
# ==========================================

tie_preds = np.array([
    [1, 0],
    [0, 1]
])

print("\nTie Breaking:")
print(hard_vote(tie_preds))

# Expected: [0 0]


# ==========================================
# STEP 12: TEST SOFT VOTING
# ==========================================

minority_probas = np.array([
    [[0.99, 0.01]],
    [[0.99, 0.01]],
    [[0.01, 0.99]]
])

print("\nSoft Voting Minority Test:")
print(soft_vote(minority_probas))

# Expected: [0]
